In [10]:
#Gizli katmanı ve çıkış katmanını kur: embedding'leri düzleştir, W1 ve b1 ile tanh, W2 ve b2 ile logits. 
#Loss'u geçen haftaki gibi elle hesapla, sonra F.cross_entropy ile aynı sonucu aldığını göster ve neden onu tercih ettiğimizi videodan anla. 
#libraries & data 
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt

words = open('names.txt', 'r').read().splitlines()

In [3]:
#character mappings; i to s or vice versa
chars = sorted(list(set(''.join(words))))
stoi = {s:i+1 for i,s in enumerate(chars)}
stoi['.'] = 0
itos = {i:s for s,i in stoi.items()}


In [4]:
block_size = 3 # context length: how many characters do we take to predict the next one?
X, Y = [], []
for w in words:
  
  context = [0] * block_size
  for ch in w + '.':
    ix = stoi[ch]
    X.append(context)
    Y.append(ix)
    context = context[1:] + [ix] # crop and append   
X = torch.tensor(X)
Y = torch.tensor(Y)

In [82]:
#27x2 lookuptable & embeddings
g = torch.Generator().manual_seed(2147483647)
C = torch.randn((27, 2), generator=g)
emb = C[X]

In [83]:
#weights and biases for hidden layer with 100 neurons 
W1 = torch.randn(6,100,generator=g)
b1 = torch.randn(100,generator=g)

In [84]:
#activation with tanh
h = torch.tanh(emb.view(-1,6) @ W1 + b1)

In [85]:
h.shape

torch.Size([228146, 100])

In [86]:
#weights and biases for output layer for 27 output neuron
W2 = torch.randn((100, 27),generator=g)
b2 = torch.randn(27,generator=g)

In [87]:
#activation for output layer
logits = h @ W2 + b2

In [88]:
#soft max
prob = logits.exp() / logits.exp().sum(1,keepdims=True)

In [89]:
prob[0].sum(),prob[1].sum() #confirmation for true softmax

(tensor(1.0000), tensor(1.0000))

In [90]:
loss = -prob[torch.arange(X.shape[0]), Y].log().mean() #loss with manual
loss

tensor(19.5052)

In [91]:
loss = F.cross_entropy(logits, Y) #loss with F.cross_entropy
loss

tensor(19.5052)